# Lesson 8: Tabular Evaluation, Feature Engineering & Pipelines

**Track:** AI Engineer — Phase 1: Data Foundations & Classical ML  
**Runtime:** Google Colab Free Tier (CPU)  
**Sources:** `.ref/ML-For-Beginners/ML-For-Beginners-main/9-Real-World/2-Debugging-ML-Models/`, `.ref/Data-Science-For-Beginners/4-Data-Science-Lifecycle/`


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance


## 1. Leak-Proof ColumnTransformer + Pipeline


In [ ]:
rng = np.random.default_rng(42)
n = 240
df = pd.DataFrame({
    'age': rng.integers(20, 65, size=n).astype(float),
    'income': rng.normal(65000, 18000, size=n),
    'region': rng.choice(['North', 'South', 'East'], size=n),
})
df.loc[rng.choice(n, 15, replace=False), 'age'] = np.nan
df['converted'] = ((df['income'] > 68000) & (df['age'].fillna(40) > 30)).astype(int)

X, y = df.drop(columns=['converted']), df['converted']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), ['age', 'income']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['region']),
])
pipe = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('5-Fold CV F1:', np.round(cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='f1'), 3))


## 2. Cohort Error Analysis & Permutation Importance


In [ ]:
pipe.fit(X_tr, y_tr)
preds = pipe.predict(X_te)
print(classification_report(y_te, preds))

eval_df = X_te.copy()
eval_df['y_true'] = y_te
eval_df['y_pred'] = preds
eval_df['error'] = (eval_df['y_true'] != eval_df['y_pred']).astype(int)
print('Error rate by region:\n', eval_df.groupby('region')['error'].mean())

imp = permutation_importance(pipe, X_te, y_te, n_repeats=10, random_state=42)
for col, score in zip(X_te.columns, imp.importances_mean):
    print(f'Permutation importance ({col}): {score:.3f}')


## Exercise
Add a new engineered feature `income_per_age` inside a custom transformer or DataFrame step and check if 5-fold CV F1 improves.
